In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pdfplumber
import nltk
import spacy
from spacy.lang.en import English
from sentence_transformers import SentenceTransformer

import sqlite3
import re
from pathlib import Path
from zipfile import ZipFile


In [2]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

nlp = spacy.load("en_core_web_sm")

nlp_2 = English()
nlp_2.add_pipe("sentencizer")

[nltk_data] Downloading package stopwords to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# import os

# # Path to your database
# db_path = "../Database/rag_documents.db"

# if os.path.exists(db_path):
#     os.unlink(db_path)  # or os.remove(db_path)
#     print(f"{db_path} deleted.")
# else:
#     print("Database file not found.")

In [3]:
DATAFILES_PATH = "../Data"
CONN = sqlite3.connect("../Database/rag_documents.db")

In [32]:
# Create DB
cursor = CONN.cursor()

# Create unified table

cursor.execute("DROP TABLE IF EXISTS documents")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_name TEXT,
    character_count INTEGER,
    word_count INTEGER,
    sentence_count INTEGER,
    token_count INTEGER,
    text TEXT
)
""")

CONN.commit()


In [33]:
def insert_page(doc_name, character_count, word_count, sentence_count, token_count, text):
    cursor = CONN.cursor()
    cursor.execute("""
        INSERT INTO pages (doc_name, character_count, word_count, sentence_count, token_count, text)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (doc_name, character_count, word_count, sentence_count, token_count, text))
    CONN.commit()

def extract_pages():
    cursor = CONN.cursor()
    cursor.execute("SELECT * FROM pages")
    return cursor.fetchall()

def clear_pages():
    cursor = CONN.cursor()
    cursor.execute("DELETE FROM pages")
    CONN.commit()

In [5]:
list(Path(DATAFILES_PATH).rglob("*"))

[WindowsPath('../Data/Cleaned_articles'),
 WindowsPath('../Data/Guidlines'),
 WindowsPath('../Data/Hospital_docs'),
 WindowsPath('../Data/Reports'),
 WindowsPath('../Data/Research_papers'),
 WindowsPath('../Data/Tables'),
 WindowsPath('../Data/Cleaned_articles/archive.zip'),
 WindowsPath('../Data/Cleaned_articles/articles'),
 WindowsPath('../Data/Cleaned_articles/articles/1.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/10.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/100.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1000.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1001.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1002.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1003.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1004.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1005.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1006.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1007.txt'),
 WindowsPath('../Da

In [4]:
def get_file_list(data_path=DATAFILES_PATH):
    data_path = Path(data_path)
    dirs = []
    files = []

    for item in data_path.rglob("*"):
        if item.is_dir():
            dirs.append(item)
        else:
            files.append(item)

    all_files = [str(i) for i in files if not (Path(i).is_dir() or str(i).endswith(".zip"))]
    pdf_files = [str(i) for i in all_files if str(i).endswith(".pdf")]
    txt_files = [str(i) for i in all_files if str(i).endswith(".txt")]
    csv_files = [str(i) for i in all_files if str(i).endswith(".csv")]
    
    return all_files, pdf_files, txt_files, csv_files

In [5]:
all_files, pdf_files, txt_files, csv_files = get_file_list()


In [49]:
def clean_txt_file(txt_file):
    doc_name = Path(txt_file).stem
    doc_type = Path(txt_file).suffix

    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as file:
        text = file.read()
    
    cleaned_text = text.capitalize()
    
    insert_page(doc_name,
            len(cleaned_text),
            len(cleaned_text.split(" ")),
            len(list(nlp_2(cleaned_text).sents)),
            len(nlp(cleaned_text)),
            cleaned_text)

    return {"doc_name": doc_name, "doc_type": doc_type, "cleaned_text": list(nlp_2(cleaned_text).sents)}

In [54]:
for file in txt_files:
    clean_txt_file(file)

In [52]:
clear_pages()

In [ ]:
pd.DataFrame(extract_pages(), columns=['id', 'doc_name', 'char_count', 'word_count', 'sent_count']).set_index('id')

In [ ]:
def clean_csv_file(csv_file):
    doc_name = Path(csv_file).stem
    doc_type = Path(csv_file).suffix
    
    df = pd.read_csv(csv_file)

    df.dropna(inplace=True)  # drop rows with any NaN values
    df = df.select_dtypes(include=[object])  # select only string columns

    def clean_text(row):
        new_list = []
        for col in row.index:  # iterate through columns
            new_list.append(f"{col.capitalize()}: {row[col]}")  
        return ". ".join(new_list)
    
    cleaned_text = df.apply(clean_text, axis=1).to_list()
    cleaned_text = ' '.join(cleaned_text)
    
    insert_page(doc_name,
                len(cleaned_text),
                len(cleaned_text.split(" ")),
                len(list(nlp_2(cleaned_text).sents)),
                len(nlp(cleaned_text)),
                cleaned_text)
    
    return {"doc_name": doc_name, "doc_type": doc_type, "cleaned_text": cleaned_text}

In [76]:
clean_csv_file(csv_files[0])

('.csv',
 'Diseases_Symptoms',
 ['Name: Panic disorder. Symptoms: Palpitations, Sweating, Trembling, Shortness of breath, Fear of losing control, Dizziness. Treatments: Antidepressant medications, Cognitive Behavioral Therapy, Relaxation Techniques',
  'Name: Vocal cord polyp. Symptoms: Hoarseness, Vocal Changes, Vocal Fatigue. Treatments: Voice Rest, Speech Therapy, Surgical Removal',
  'Name: Turner syndrome. Symptoms: Short stature, Gonadal dysgenesis, Webbed neck, Lymphedema. Treatments: Growth hormone therapy, Estrogen replacement therapy, Cardiac and renal evaluations',
  'Name: Cryptorchidism. Symptoms: Absence or undescended testicle(s), empty scrotum, smaller or underdeveloped testicle(s), inguinal hernia, abnormal positioning of the testicle(s) (higher in the groin area). Treatments: Observation and monitoring (in cases of mild or transient cryptorchidism), hormone therapy (to stimulate testicular descent), surgical intervention (orchiopexy) to reposition the testicle(s) into

In [69]:
str(clean_csv_file(csv_files[0])[2])

'[\'Name: Panic disorder. Symptoms: Palpitations, Sweating, Trembling, Shortness of breath, Fear of losing control, Dizziness. Treatments: Antidepressant medications, Cognitive Behavioral Therapy, Relaxation Techniques\', \'Name: Vocal cord polyp. Symptoms: Hoarseness, Vocal Changes, Vocal Fatigue. Treatments: Voice Rest, Speech Therapy, Surgical Removal\', \'Name: Turner syndrome. Symptoms: Short stature, Gonadal dysgenesis, Webbed neck, Lymphedema. Treatments: Growth hormone therapy, Estrogen replacement therapy, Cardiac and renal evaluations\', \'Name: Cryptorchidism. Symptoms: Absence or undescended testicle(s), empty scrotum, smaller or underdeveloped testicle(s), inguinal hernia, abnormal positioning of the testicle(s) (higher in the groin area). Treatments: Observation and monitoring (in cases of mild or transient cryptorchidism), hormone therapy (to stimulate testicular descent), surgical intervention (orchiopexy) to reposition the testicle(s) into the scrotum, if necessary, pe

In [33]:
def classify_page(text):
    doc = nlp(text)
    lowered = text.lower()
    num_lines = len(text.splitlines())

    # Heuristic 1: Sentence length
    sents = list(doc.sents)
    avg_len = sum(len(sent.text) for sent in sents) / max(1, len(sents))

    # Heuristic 2: POS distribution
    pos_counts = doc.count_by(spacy.attrs.POS)
    num_verbs = pos_counts.get(doc.vocab.strings["VERB"], 0)
    num_nouns = pos_counts.get(doc.vocab.strings["NOUN"], 0)
    num_propns = pos_counts.get(doc.vocab.strings["PROPN"], 0)

    # Heuristic 3: Numbers/digits
    num_digits = sum(c.isdigit() for c in text)
    digit_ratio = num_digits / max(1, len(text))

    # ------------------ RULES ------------------ #

    # References / Bibliography → many proper nouns + numbers + citations
    if ("references" in lowered or "bibliography" in lowered) or \
       (num_propns > 15 and num_verbs < 5 and digit_ratio > 0.05):
        return "reference"

    # TOC → many short lines, dotted leaders, section/chapter keywords
    if ("table of contents" in lowered or "contents" in lowered) or \
       (re.search(r"\.{5,}\s*\d+", text) and num_verbs < 5 and num_lines > 5):
        return "toc"

    # Acknowledgement → gratitude words + relatively short text
    if any(word in lowered for word in ["thank", "thankful","grateful", "acknowledgements" ,"acknowledgement", "debt of gratitude", "appreciation", "appreciations"]):
        if len(text) < 1000:  # avoid false positives in main body
            return "acknowledgement"

    # If nothing matched → assume content
    return "content"

In [ ]:
def clean_pdf_file(pdf_file, first_page=None, last_page=None):
    doc_name = Path(pdf_file).stem
    doc_type = Path(pdf_file).suffix
    results = []
    
    with pdfplumber.open(pdf_file) as pdf:
        if first_page or last_page:
            pages = pdf.pages[first_page:last_page]
            for page_number, page in enumerate(pages, start=first_page or 0):
                if text := page.extract_text(x_tolerance=1, y_tolerance=1).replace('\n', ' '):
                    if text.strip():  # ignore empty pages
                        results.append((page_number, text))
        else:
            for page_number, page in enumerate(pdf.pages):
                if text := page.extract_text(x_tolerance=1, y_tolerance=1).replace('\n', ' '):
                    if text.strip():  # ignore empty pages
                        if classify_page(text) not in ["reference", "toc", "acknowledgement"]:
                            results.append((page_number, text))

    cleaned_results = [text for page_number, text in results]
    return {"doc_name": doc_name, "doc_type": doc_type, "cleaned_text": cleaned_results}

In [37]:
pdf_files

['..\\Data\\Guidlines\\EPR-3_Asthma_Full_Report_2007.pdf',
 '..\\Data\\Guidlines\\multivitamin-mineral-suppl-cvd-cancer-prev-final-recommendation.pdf',
 '..\\Data\\Reports\\CaseReportExample.pdf',
 '..\\Data\\Research_papers\\Emerging trends and global collaboration in Paclitaxel resistance research for breast cancer a comprehensive bibliometric study.pdf',
 '..\\Data\\Research_papers\\Impact of cycling exercise in patients.pdf']

In [ ]:
data = pd.read_csv(csv_files[0])
len(data)

400

In [21]:
def split_rows(num):
    items = list(range(0, num+1, 50))
    print([[items[i], items[i+1]] for i in range(len(items)-1)])


split_rows(124)

[[0, 50], [50, 100]]


In [ ]:
df_trial = pd.read_csv(csv_files[2])
df_trial.head()

df_trial = df_trial.select_dtypes(include=[object]).head()

def clean_text(row):
    new_list = []
    for col in row.index:  # iterate through columns
        new_list.append(f"{col.capitalize()}: {row[col]}")  
    return ". ".join(new_list)

df_trial.apply(clean_text, axis=1).to_list()



['Label: Psoriasis. Text: I have been experiencing a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.',
 'Label: Psoriasis. Text: My skin has been peeling, especially on my knees, elbows, and scalp. This peeling is often accompanied by a burning or stinging sensation.',
 'Label: Psoriasis. Text: I have been experiencing joint pain in my fingers, wrists, and knees. The pain is often achy and throbbing, and it gets worse when I move my joints.',
 'Label: Psoriasis. Text: There is a silver like dusting on my skin, especially on my lower back and scalp. This dusting is made up of small scales that flake off easily when I scratch them.',
 'Label: Psoriasis. Text: My nails have small dents or pits in them, and they often feel inflammatory and tender to the touch. Even there are minor rashes on my arms.']